# Análisis y Visualización de Datos

Notebook completo con 9 tipos de visualizaciones usando Pandas y Plotly, aplicando principios de diseño de visualización de datos.

## Importaciones

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## Carga de Datos y Exploración Inicial

In [ ]:
df = pd.read_csv('data/student_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"Total records: {len(df)}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
df.head(10)

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
df.info()

In [ ]:
df.describe()

## Color Palette & Design Configuration

Using a validated accessibility-safe color palette with consistent design principles applied to all visualizations.

In [ ]:
# Color palette - validated for accessibility and CVD safety
COLOR_PALETTE = {
    'blue': '#2a78d6',
    'orange': '#eb6834',
    'aqua': '#1baf7a',
    'yellow': '#eda100',
    'magenta': '#e87ba4',
    'green': '#008300',
    'violet': '#4a3aa7',
    'red': '#e34948'
}

# Define a consistent template for all charts
CHART_TEMPLATE = 'plotly_white'
CHART_SURFACE = '#fcfcfb'

---
# 1. BAR CHART

**Variable(s):** `school` (categorical) - Count of students per school

**Why this chart:** Bar charts compare values across categorical groups. School is a nominal categorical variable (GP, MS), and counting students per school reveals the composition across these categories.

**Design Principles Applied:**
- **Contrast:** Blue bars stand out prominently against the white background; muted gridlines recede
- **Hierarchy:** Title at top guides eye first, then to bar heights showing magnitude
- **Proximity:** Count labels positioned directly above bars for easy reading
- **Simplicity:** Clean template, no 3D effects, minimal decoration
- **Similarity:** Consistent blue used for categorical comparisons

In [ ]:
school_counts = df['school'].value_counts().reset_index()
school_counts.columns = ['School', 'Count']

fig1 = px.bar(
    school_counts,
    x='School',
    y='Count',
    color_discrete_sequence=[COLOR_PALETTE['blue']],
    title='Student Distribution by School Type',
    labels={'School': 'School Type', 'Count': 'Number of Students'},
    template=CHART_TEMPLATE,
    text='Count'
)

fig1.update_traces(textposition='outside', textfont=dict(size=12, color='#0b0b0b'))
fig1.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=False,
    height=500,
    plot_bgcolor=CHART_SURFACE,
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    xaxis=dict(showgrid=False)
)
fig1.show()

---
# 2. LINE CHART

**Variable(s):** `G1`, `G2`, `G3` (three numeric variables representing grades across periods)

**Why this chart:** Line charts show progression and change over time or sequence. The three grade columns represent a temporal sequence (first period, second period, final grade), making a line chart ideal for visualizing how average student performance evolves across the school year.

**Design Principles Applied:**
- **Contrast:** Orange line with 3px thickness stands out against white background; semi-transparent fill adds depth without clutter
- **Hierarchy:** Title emphasizes progression pattern, grade periods clearly ordered along x-axis
- **Proximity:** Markers positioned at data points for precise values
- **Simplicity:** Minimal gridlines, no multiple series or legends needed
- **Similarity:** Orange used consistently for progression/change encoding

In [ ]:
grade_progression = pd.DataFrame({
    'Period': ['G1 (First)', 'G2 (Second)', 'G3 (Final)'],
    'Average Grade': [df['G1'].mean(), df['G2'].mean(), df['G3'].mean()]
})

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=grade_progression['Period'],
    y=grade_progression['Average Grade'],
    mode='lines+markers',
    name='Average Grade',
    line=dict(color=COLOR_PALETTE['orange'], width=3),
    marker=dict(size=12, color=COLOR_PALETTE['orange'], symbol='circle'),
    fill='tozeroy',
    fillcolor='rgba(235, 104, 52, 0.15)'
))

fig2.update_layout(
    title='Average Student Grade Progression Across Periods',
    xaxis_title='Grade Period',
    yaxis_title='Average Grade',
    template=CHART_TEMPLATE,
    plot_bgcolor=CHART_SURFACE,
    height=500,
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    hovermode='x unified',
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5, range=[0, 20]),
    xaxis=dict(showgrid=False),
    showlegend=False
)
fig2.show()

---
# 3. PIE CHART

**Variable(s):** `internet` (categorical binary - yes/no for internet access at home)

**Why this chart:** Pie charts show composition and parts of a whole as percentages. Internet access is a binary categorical variable where displaying the proportion of students with/without home internet shows this binary split as a clear visual composition.

**Design Principles Applied:**
- **Contrast:** Aqua slice stands out as the primary category; muted gray for secondary
- **Hierarchy:** Percentages are primary information; labels positioned around the donut
- **Proximity:** Labels positioned directly adjacent to their corresponding slices
- **Simplicity:** Donut format (not full pie) reduces visual weight; no 3D effects
- **Similarity:** Aqua used consistently for positive/yes categories

In [ ]:
internet_dist = df['internet'].value_counts().reset_index()
internet_dist.columns = ['Internet Access', 'Count']

# Map yes/no to readable labels
labels_map = {'yes': 'Has Internet', 'no': 'No Internet'}
internet_dist['Label'] = internet_dist['Internet Access'].map(labels_map)

fig3 = go.Figure(data=[go.Pie(
    labels=internet_dist['Label'],
    values=internet_dist['Count'],
    hole=0.4,
    marker=dict(
        colors=[COLOR_PALETTE['aqua'], '#c3c2b7'],
        line=dict(color=CHART_SURFACE, width=2)
    ),
    textinfo='label+percent',
    textfont=dict(size=12, color='#0b0b0b'),
    hovertemplate='%{label}<br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)])

fig3.update_layout(
    title='Home Internet Access Distribution',
    template=CHART_TEMPLATE,
    height=500,
    title_font_size=16,
    plot_bgcolor=CHART_SURFACE,
    showlegend=False,
    font=dict(family='system-ui', size=11)
)
fig3.show()

---
# 4. HISTOGRAM

**Variable(s):** `age` (numeric continuous - student ages in years)

**Why this chart:** Histograms display the distribution of a single continuous numeric variable using bins. Age reveals how students are distributed across age ranges—whether the cohort is concentrated at certain ages or spread uniformly.

**Design Principles Applied:**
- **Contrast:** Yellow bars with dark borders stand out strongly against white background
- **Hierarchy:** Title describes the distribution shape; bar heights show frequency magnitude
- **Proximity:** Bin boundaries clearly marked on x-axis
- **Simplicity:** Equal-width bins, no 3D effects or unnecessary decoration
- **Similarity:** Yellow used consistently for numeric distributions

In [ ]:
fig4 = px.histogram(
    df,
    x='age',
    nbins=10,
    color_discrete_sequence=[COLOR_PALETTE['yellow']],
    title='Age Distribution of Students',
    labels={'age': 'Age (years)', 'count': 'Number of Students'},
    template=CHART_TEMPLATE
)

fig4.update_traces(marker_line=dict(width=1, color='#52514e'))
fig4.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=False,
    height=500,
    plot_bgcolor=CHART_SURFACE,
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    xaxis=dict(showgrid=False)
)
fig4.show()

---
# 5. SCATTER PLOT

**Variable(s):** `studytime` (numeric ordinal 1-4) vs `G3` (numeric continuous - final grade)

**Why this chart:** Scatter plots reveal relationships and correlations between two numeric variables. Plotting study time against final grades shows whether students who study more tend to achieve higher grades—a key insight for student success factors.

**Design Principles Applied:**
- **Contrast:** Magenta markers (≥8px) stand out against white background; transparency shows overlaps
- **Hierarchy:** Title states the relationship being tested; axes clearly labeled
- **Proximity:** Both axes positioned close to the data cloud
- **Simplicity:** One series only, no legend needed; gridlines recessive
- **Similarity:** Magenta used consistently for relationship/correlation encoding

In [ ]:
fig5 = px.scatter(
    df,
    x='studytime',
    y='G3',
    color_discrete_sequence=[COLOR_PALETTE['magenta']],
    title='Relationship: Study Time vs Final Grade',
    labels={
        'studytime': 'Study Time (1=<2hrs, 2=2-5hrs, 3=5-10hrs, 4=>10hrs)', 
        'G3': 'Final Grade (0-20)'
    },
    template=CHART_TEMPLATE,
    opacity=0.6
)

fig5.update_traces(
    marker=dict(size=10, line=dict(width=0.5, color=CHART_SURFACE)),
    selector=dict(mode='markers')
)
fig5.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=False,
    height=500,
    plot_bgcolor=CHART_SURFACE,
    xaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    hovermode='closest'
)
fig5.show()

---
# 6. BOX PLOT

**Variable(s):** `sex` (categorical - M/F) vs `G3` (numeric continuous - final grade)

**Why this chart:** Box plots compare distributions of a numeric variable across categorical groups. Sex is a binary categorical variable; comparing grade distributions by gender reveals whether one group has systematically higher or lower academic performance, and differences in grade variability.

**Design Principles Applied:**
- **Contrast:** Green boxes stand out; muted gridlines recede
- **Hierarchy:** Title emphasizes the comparison; box positions guide eye across groups
- **Proximity:** Category labels positioned directly below boxes
- **Simplicity:** Minimal ink—only quartiles and outliers shown; clean box format
- **Similarity:** Green used consistently for group comparisons

In [ ]:
fig6 = px.box(
    df,
    x='sex',
    y='G3',
    color_discrete_sequence=[COLOR_PALETTE['green']],
    title='Final Grade Distribution by Gender',
    labels={'sex': 'Gender (M=Male, F=Female)', 'G3': 'Final Grade'},
    template=CHART_TEMPLATE,
    points=False
)

fig6.update_traces(
    marker=dict(size=6),
    line=dict(width=2)
)
fig6.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=False,
    height=500,
    plot_bgcolor=CHART_SURFACE,
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    xaxis=dict(showgrid=False)
)
fig6.show()

---
# 7. VIOLIN PLOT

**Variable(s):** `sex` (categorical - M/F) vs `age` (numeric continuous)

**Why this chart:** Violin plots reveal the full distribution shape (including multimodality) across groups better than box plots. Age by gender shows whether males and females have different age distributions and whether those distributions are symmetric, skewed, or bimodal.

**Design Principles Applied:**
- **Contrast:** Violet fill with clear borders stands out; symmetric mirror shape emphasizes distribution form
- **Hierarchy:** Title emphasizes distribution shape comparison; vertical axis shows magnitude
- **Proximity:** Gender labels positioned directly below violins
- **Simplicity:** Clean contours only, no box clutter; mean line adds reference without noise
- **Similarity:** Violet used consistently for distribution-shape encoding

In [ ]:
fig7 = px.violin(
    df,
    x='sex',
    y='age',
    color_discrete_sequence=[COLOR_PALETTE['violet']],
    title='Age Distribution Shape by Gender (Violin Plot)',
    labels={'sex': 'Gender (M=Male, F=Female)', 'age': 'Age (years)'},
    template=CHART_TEMPLATE,
    box=False,
    points=False
)

fig7.update_traces(
    line=dict(width=1.5),
    meanline_visible=True
)
fig7.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=False,
    height=500,
    plot_bgcolor=CHART_SURFACE,
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    xaxis=dict(showgrid=False)
)
fig7.show()

---
# 8. DENSITY PLOT (KDE)

**Variable(s):** `G3` (numeric continuous - final grade)

**Why this chart:** Density plots (Kernel Density Estimation) show the smooth probability distribution of a numeric variable. Unlike histograms with bin artifacts, density plots reveal whether grades follow a normal distribution or are skewed, providing insight into the underlying grade distribution shape.

**Design Principles Applied:**
- **Contrast:** Red line (3px) stands out; semi-transparent fill adds visual weight without obscuring the curve
- **Hierarchy:** Title explains the metric; smooth curve is the focal element
- **Proximity:** Axis labels positioned close to axes
- **Simplicity:** Clean curve only, no histogram bins or unnecessary decoration
- **Similarity:** Red used consistently for single-variable density encoding

In [ ]:
# Create density data using KDE
kde_data = stats.gaussian_kde(df['G3'])
x_range = range(int(df['G3'].min()), int(df['G3'].max()) + 1)
density_values = kde_data(x_range)

fig8 = go.Figure()

fig8.add_trace(go.Scatter(
    x=list(x_range),
    y=density_values,
    fill='tozeroy',
    fillcolor='rgba(227, 73, 72, 0.25)',
    line=dict(color=COLOR_PALETTE['red'], width=3),
    name='Density',
    hovertemplate='Grade: %{x}<br>Density: %{y:.4f}<extra></extra>'
))

fig8.update_layout(
    title='Probability Density Distribution of Final Grades (KDE)',
    xaxis_title='Final Grade',
    yaxis_title='Density',
    template=CHART_TEMPLATE,
    plot_bgcolor=CHART_SURFACE,
    height=500,
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=False,
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    xaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5)
)
fig8.show()

---
# 9. HEATMAP (Correlation Matrix)

**Variable(s):** Multiple numeric columns - `age`, `Medu`, `Fedu`, `studytime`, `failures`, `health`, `G1`, `G2`, `G3`

**Why this chart:** Heatmaps visualize correlations between multiple numeric variables simultaneously. Each cell shows correlation strength as color intensity, making patterns across many relationships visible at once—ideal for identifying which factors most influence final grades.

**Design Principles Applied:**
- **Contrast:** Diverging blue-to-red palette makes positive (red) and negative (blue) correlations immediately apparent; gray midpoint recedes
- **Hierarchy:** Title explains the matrix; cell color saturation guides eye to strongest correlations first
- **Proximity:** Variable labels positioned closely on both axes; values placed in cell centers
- **Simplicity:** Grid separates cells clearly; no 3D or unnecessary decoration
- **Similarity:** Diverging palette applied consistently to correlation magnitude across all cells

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['age', 'Medu', 'Fedu', 'studytime', 'failures', 'health', 'G1', 'G2', 'G3']
correlation_matrix = df[numeric_cols].corr()

fig9 = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=numeric_cols,
    y=numeric_cols,
    colorscale=[
        [0, '#184f95'],      # dark blue (strong negative)
        [0.25, '#5598e7'],   # light blue
        [0.5, '#f0efec'],    # neutral gray
        [0.75, '#eb6834'],   # orange
        [1, '#e34948']       # red (strong positive)
    ],
    zmid=0,
    text=correlation_matrix.values,
    texttemplate='%{text:.2f}',
    textfont=dict(size=10, color='#0b0b0b'),
    colorbar=dict(title='Correlation', len=0.7),
    hovertemplate='%{y} vs %{x}<br>Correlation: %{z:.3f}<extra></extra>'
))

fig9.update_layout(
    title='Correlation Matrix: All Numeric Variables',
    xaxis_title='Variables',
    yaxis_title='Variables',
    template=CHART_TEMPLATE,
    height=600,
    width=700,
    title_font_size=16,
    xaxis_tickangle=-45,
    plot_bgcolor=CHART_SURFACE,
    yaxis=dict(tickfont=dict(size=10)),
    xaxis=dict(tickfont=dict(size=10))
)
fig9.show()

---
## Summary of Design Principles Across All Charts

**Contrast:** Each chart uses one primary color for the main data element, with muted secondary colors and gridlines that recede into the background. This ensures the key insight stands out immediately.

**Hierarchy:** Titles are positioned at the top in larger font to guide the eye first. Main data is encoded in the chart body. Supporting details (axis labels, gridlines) are progressively de-emphasized.

**Proximity:** Related elements are grouped—axis labels positioned near their axes, legends placed near data, category labels below or beside corresponding marks.

**Similarity:** The same variable type uses the same color across all charts (blue for categories, orange for progression, aqua for binary, etc.), creating visual consistency and making patterns memorable.

**Simplicity:** All charts use the clean `plotly_white` template with minimal gridlines, no 3D effects, no redundant legends for single series, and thin mark weights. The focus is on data, not decoration.